# 05 - Model Evaluation and Final Artifact

Focused evaluation of the measured production candidate. No full baseline comparison is rerun.

## 1. Load existing experiment results
Question: what measured Step 04 evidence informs selection?

In [ ]:
from pathlib import Path
import pandas as pd
from src.models.evaluate import STEP4_VALIDATION, STEP4_CV, evaluate_candidate, save_production_artifact
ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists(): ROOT = ROOT.parent
STEP4_VALIDATION, STEP4_CV

## 2. Evaluation setup
Question: does focused evaluation retain the exact stratified Step 04 holdout?

In [ ]:
train = pd.read_csv(ROOT / 'data/processed/train_clean.csv')
result = evaluate_candidate(train)
len(result['X_train']), len(result['X_valid'])

## 3. Confusion matrix
Question: how do actual and predicted outcomes compare for Gradient Boosting?

In [ ]:
result['matrix']
ROOT / 'web/assets/generated/final_confusion_matrix.png'

## 4. ROC analysis
Question: what does ROC-AUC indicate about ranking discrimination on this holdout?

In [ ]:
STEP4_VALIDATION[['Model', 'ROC-AUC']]
ROOT / 'web/assets/generated/roc_curve_comparison.png'

## 5. Precision/Recall analysis
Question: what observed precision/recall balance do the fixed baselines have?

In [ ]:
STEP4_VALIDATION[['Model', 'Precision', 'Recall', 'F1']]
ROOT / 'web/assets/generated/precision_recall_comparison.png'

## 6. Cross-validation stability
Question: how much did fixed configurations vary across five stratified folds?

In [ ]:
STEP4_CV
ROOT / 'web/assets/generated/cv_stability.png'

## 7. Error analysis
Question: which validation subgroups show different descriptive error rates?

In [ ]:
result['error_rates']['age'], result['error_rates']['cryo']
ROOT / 'web/assets/generated/error_rate_by_age_group.png'

## 8. Feature importance
Question: what does the existing Random Forest internal importance chart describe?

In [ ]:
ROOT / 'web/assets/generated/feature_importance.png'
'Importance is model-internal, not causal.'

## 9. Production candidate
Question: which fixed configuration has the strongest recorded ROC-AUC and CV F1 evidence?

In [ ]:
STEP4_VALIDATION.loc[STEP4_VALIDATION['Model'].eq('Gradient Boosting')]
STEP4_CV.loc[STEP4_CV['Model'].eq('Gradient Boosting')]

## 10. Save final pipeline
Question: can the selected pipeline be refit on all labelled data and persisted once?

In [ ]:
artifact, metadata = save_production_artifact(train, ROOT / 'model')
artifact, metadata

## 11. Final evaluation summary
Gradient Boosting is the documented production candidate. `MODEL_RESULTS.md` and `docs/MODEL_CARD.md` contain metrics, limitations, and reproducibility notes; holdout results do not guarantee production performance.